<a href="https://colab.research.google.com/github/ericsonsneresti-dev/Rede-neural-/blob/main/MVPfraudesdigitais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# 1. Dataset Ampliado e Mais Robusto para o MVP
dados_treino = {
    'texto': [
        # Fraudes / Golpes
        "Seu token de acesso expirou, clique aqui para atualizar sua conta imediatamente.",
        "Parabéns! Você ganhou um sorteio de 5000 reais, clique no link para resgatar.",
        "Olá pai, troquei de número, me manda um PIX urgente para pagar uma conta.",
        "URGENTE: Bloqueio preventivo no seu CPF. Regularize agora via link.",
        "Oportunidade única de investimento com retorno financeiro garantido em 24h.",
        "Sua conta bancária foi bloqueada por acesso suspeito. Clique para desbloquear.",
        "Cartão clonado detectado em compra de R$ 3.490,00. Confirme sua senha aqui.",

        # Legítimos
        "Reunião de alinhamento de projeto marcada para amanhã às 14h.",
        "Segue em anexo o relatório mensal de atividades da diretoria.",
        "Lembrete: sua fatura com vencimento dia 10 está disponível no aplicativo.",
        "Favor confirmar o recebimento do e-mail com a ata da última reunião.",
        "Confirmação de agendamento de consulta médica para a próxima terça-feira.",
        "Seu pedido #4892 foi enviado e está a caminho do endereço de entrega.",
        "Comunicado interno: manutenção programada dos servidores da STI neste fim de semana."
    ],
    'rotulo': [
        '⚠️ Fraude / Golpe', '⚠️ Fraude / Golpe', '⚠️ Fraude / Golpe',
        '⚠️ Fraude / Golpe', '⚠️ Fraude / Golpe', '⚠️ Fraude / Golpe', '⚠️ Fraude / Golpe',
        '✅ Legítimo', '✅ Legítimo', '✅ Legítimo',
        '✅ Legítimo', '✅ Legítimo', '✅ Legítimo', '✅ Legítimo'
    ]
}

df = pd.DataFrame(dados_treino)

# 2. Treinamento do Pipeline de Machine Learning
# Usando n-grams (ngram_range=(1,2)) para capturar tanto palavras isoladas quanto expressões compostas
modelo = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1),
    LogisticRegression(max_iter=500)
)
modelo.fit(df['texto'], df['rotulo'])
print("Modelo de IA treinado com sucesso com dataset expandido!")

# 3. Lógica de Predição com Detalhes de Confiança e Termos-Chave
def prever_fraude(mensagem):
    if not mensagem or not mensagem.strip():
        return "Por favor, digite ou selecione uma mensagem para análise.", "N/A"

    predicao = modelo.predict([mensagem])[0]
    probabilidades = modelo.predict_proba([mensagem])[0]
    classes = modelo.classes_

    # Mapeia as probabilidades para cada classe
    prob_dict = {cls: prob * 100 for cls, prob in zip(classes, probabilidades)}
    confianca_max = max(probabilidades) * 100

    # Formatando a saída detalhada
    resultado_txt = f"### Classificação Final: **{predicao}**\n\n"
    resultado_txt += f"* **Nível de Confiança:** `{confianca_max:.2f}%`\n"
    resultado_txt += f"* **Probabilidade (Fraude / Golpe):** `{prob_dict.get('⚠️ Fraude / Golpe', 0):.2f}%`\n"
    resultado_txt += f"* **Probabilidade (Legítimo):** `{prob_dict.get('✅ Legítimo', 0):.2f}%`"

    return resultado_txt

# 4. Textos de Apresentação Acadêmica Aprimorados
texto_intro = """
# 🛡️ MVP: Detector Inteligente de Fraudes Digitais e Engenharia Social
- **Contexto Acadêmico:** Tópicos em Segurança Cibernética (Redes Neurais e Aprendizado de Máquina Aplicados à Defesa)
- **Tecnologias:** Python, Scikit-Learn (TF-IDF + Regressão Logística), Gradio.
- **Instruções:** Cole qualquer mensagem suspeita (SMS, WhatsApp, E-mail) ou clique nos exemplos prontos abaixo para testar o classificador em tempo real.
"""

# 5. Interface Web Otimizada com Gradio
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(texto_intro)

    with gr.Row():
        with gr.Column():
            input_texto = gr.Textbox(
                lines=4,
                label="Mensagem para Análise",
                placeholder="Ex: Seu token expirou, clique aqui..."
            )
            with gr.Row():
                btn_limpar = gr.Button("Limpar", variant="secondary")
                btn_enviar = gr.Button("Analisar Mensagem", variant="primary")

        with gr.Column():
            output_resultado = gr.Markdown(label="Resultado da Análise")

    # Exemplos rápidos para facilitar os testes na apresentação
    gr.Examples(
        examples=[
            ["Seu token de segurança expirou. Clique no link para atualizar."],
            ["Olá pai, troquei de número, me manda um PIX urgente."],
            ["Reunião de alinhamento de projeto marcada para amanhã às 14h."],
            ["Parabéns! Você foi selecionado para resgatar um prêmio de R$ 5.000."]
        ],
        inputs=input_texto
    )

    # Ações dos botões
    btn_enviar.fn = prever_fraude
    btn_enviar.inputs = [input_texto]
    btn_enviar.outputs = [output_resultado]

    btn_enviar.click(fn=prever_fraude, inputs=input_texto, outputs=output_resultado)
    btn_limpar.click(fn=lambda: ("", ""), inputs=None, outputs=[input_texto, output_resultado])

# Rodar a aplicação atualizada
demo.launch(share=True, debug=True)

Modelo de IA treinado com sucesso com dataset expandido!


/tmp/ipykernel_1578/2255593179.py:77: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4703fd461a80d301d0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
